**Install the required libraries**

In [4]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Check the installation**

In [5]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)
print("Installation successful!")

Scikit-learn version: 1.9.0
Installation successful!


**Create the sample document**

In [6]:

sample_text = """
Artificial Intelligence (AI) enables machines to perform
tasks that normally require human intelligence.

Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Natural Language Processing (NLP) helps computers
understand, process, and generate human language.

Information retrieval finds relevant documents or passages
based on a user's query.

TF-IDF stands for Term Frequency-Inverse Document Frequency.
It helps identify words that are important in a document
relative to a collection of documents.
"""

print("Sample document created successfully!")
print(sample_text)


Sample document created successfully!

Artificial Intelligence (AI) enables machines to perform
tasks that normally require human intelligence.

Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Natural Language Processing (NLP) helps computers
understand, process, and generate human language.

Information retrieval finds relevant documents or passages
based on a user's query.

TF-IDF stands for Term Frequency-Inverse Document Frequency.
It helps identify words that are important in a document
relative to a collection of documents.



**Split the document into passages**

In [7]:

import re

# Normalize line breaks and split into sentences
clean_text = " ".join(sample_text.split())
sentences = re.split(r"(?<=[.!?])\s+", clean_text)

chunks = [
    " ".join(sentences[i:i + 2])
    for i in range(0, len(sentences), 2)
]

print("Total passages:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\nPassage {i}:")
    print(chunk)


Total passages: 3

Passage 1:
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence. Machine Learning is a branch of AI that allows computers to learn patterns from data and make predictions.

Passage 2:
Natural Language Processing (NLP) helps computers understand, process, and generate human language. Information retrieval finds relevant documents or passages based on a user's query.

Passage 3:
TF-IDF stands for Term Frequency-Inverse Document Frequency. It helps identify words that are important in a document relative to a collection of documents.


In [8]:

# Split the document into paragraphs
chunks = [
    paragraph.strip()
    for paragraph in sample_text.strip().split("\n\n")
    if paragraph.strip()
]

print("Total passages:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\nPassage {i}:")
    print(chunk)


Total passages: 5

Passage 1:
Artificial Intelligence (AI) enables machines to perform
tasks that normally require human intelligence.

Passage 2:
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Passage 3:
Natural Language Processing (NLP) helps computers
understand, process, and generate human language.

Passage 4:
Information retrieval finds relevant documents or passages
based on a user's query.

Passage 5:
TF-IDF stands for Term Frequency-Inverse Document Frequency.
It helps identify words that are important in a document
relative to a collection of documents.


**Rebuild TF-IDF**

In [9]:

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(chunks)

print("TF-IDF model rebuilt successfully!")
print("Number of passages:", tfidf_matrix.shape[0])


TF-IDF model rebuilt successfully!
Number of passages: 5


**Verify the TF-IDF model**

In [10]:
print("Number of passages:", tfidf_matrix.shape[0])

Number of passages: 5


**Retrieve the relevant passage**

In [11]:

question = "What is Machine Learning?"

question_vector = vectorizer.transform([question])

similarity_scores = cosine_similarity(
    question_vector, tfidf_matrix
).flatten()

best_index = similarity_scores.argmax()

print("Question:", question)
print("\nRetrieved passage:")
print(chunks[best_index])
print("\nSimilarity score:", round(similarity_scores[best_index], 3))


Question: What is Machine Learning?

Retrieved passage:
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Similarity score: 0.441


**Create the question-answering function**

In [12]:

def answer_question(question):
    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector, tfidf_matrix
    ).flatten()

    best_index = scores.argmax()
    best_score = scores[best_index]

    # Reject questions with no relevant matching words
    if best_score == 0 or best_score < 0.10:
        print("\nAnswer: Information not found in provided document.")
        return

    print("\nAnswer:")
    print(chunks[best_index])

    print("\nSupporting source passage:")
    print("Document: sample_document")
    print("Passage number:", best_index + 1)
    print(chunks[best_index])

    print("\nRelevance score:", round(best_score, 3))


print("Question-answering function created successfully!")


Question-answering function created successfully!


**Test the assistant**

In [13]:
answer_question("What is Machine Learning?")


Answer:
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Supporting source passage:
Document: sample_document
Passage number: 2
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Relevance score: 0.441


In [14]:
answer_question("Who won the FIFA World Cup in 2022?")


Answer: Information not found in provided document.


**Make it interactive**

In [15]:

while True:
    question = input("Ask a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Assistant stopped.")
        break

    answer_question(question)
    print("-" * 50)


Ask a question (type 'exit' to stop):  exit


Assistant stopped.


**Sample document creation**

In [16]:
with open("sample_document.txt", "w", encoding="utf-8") as file:
    file.write(sample_text)

print("sample_document.txt saved successfully!")

sample_document.txt saved successfully!


**document_qa.py creation**

In [25]:

source_code = r'''
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DOCUMENT = Path("sample_document.txt")
THRESHOLD = 0.10


def load_document():
    if not DOCUMENT.exists():
        raise FileNotFoundError("sample_document.txt was not found.")
    return DOCUMENT.read_text(encoding="utf-8")


def split_into_passages(text):
    return [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]


def build_retriever(passages):
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(passages)
    return vectorizer, matrix


def answer_question(question, passages, vectorizer, matrix):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, matrix).flatten()

    best_index = scores.argmax()
    best_score = scores[best_index]

    if best_score == 0 or best_score < THRESHOLD:
        print("\nAnswer: Information not found in provided document.")
        return

    print("\nAnswer:")
    print(passages[best_index])

    print("\nSupporting source passage:")
    print(f"Document: {DOCUMENT.name}")
    print(f"Passage number: {best_index + 1}")
    print(passages[best_index])

    print(f"\nRelevance score: {best_score:.3f}")


def main():
    text = load_document()
    passages = split_into_passages(text)

    if not passages:
        print("The document contains no readable passages.")
        return

    vectorizer, matrix = build_retriever(passages)

    print("DOCUMENT QUESTION-ANSWERING ASSISTANT")
    print("Type 'exit' to stop.")

    while True:
        question = input("\nAsk a question: ").strip()

        if question.lower() == "exit":
            print("Assistant stopped.")
            break

        if not question:
            print("Please enter a question.")
            continue

        answer_question(question, passages, vectorizer, matrix)


if __name__ == "__main__":
    main()
'''

with open("document_qa.py", "w", encoding="utf-8") as file:
    file.write(source_code)

print("document_qa.py created successfully!")


document_qa.py created successfully!


**requirements.txt creation**

In [26]:
with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write("scikit-learn\n")

print("requirements.txt created successfully!")

requirements.txt created successfully!


**Check all required files**

In [31]:

from pathlib import Path

files = [
    "Document_QA_Assistant.ipynb",
    "document_qa.py",
    "sample_document.txt",
    "requirements.txt",
    "README.md"
]

print("PROJECT FILE VERIFICATION\n")

for filename in files:
    status = "FOUND" if Path(filename).exists() else "MISSING"
    print(f"{filename}: {status}")


PROJECT FILE VERIFICATION

Document_QA_Assistant.ipynb: FOUND
document_qa.py: FOUND
sample_document.txt: FOUND
requirements.txt: FOUND
README.md: FOUND


**DOCUMENT QUESTION-ANSWERING ASSISTANT**

In [33]:
%run document_qa.py

DOCUMENT QUESTION-ANSWERING ASSISTANT
Type 'exit' to stop.



Ask a question:  what is MAchine Learning?



Answer:
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Supporting source passage:
Document: sample_document.txt
Passage number: 2
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Relevance score: 0.441



Ask a question:  who invented ML?



Answer: Information not found in provided document.



Ask a question:  exit


Assistant stopped.


In [34]:
from pathlib import Path

print("Sample document exists:", Path("sample_document.txt").exists())

Sample document exists: True


In [35]:
print("Total passages:", len(chunks))

for i, passage in enumerate(chunks, start=1):
    print(f"\nPassage {i}:")
    print(passage)

Total passages: 5

Passage 1:
Artificial Intelligence (AI) enables machines to perform
tasks that normally require human intelligence.

Passage 2:
Machine Learning is a branch of AI that allows computers
to learn patterns from data and make predictions.

Passage 3:
Natural Language Processing (NLP) helps computers
understand, process, and generate human language.

Passage 4:
Information retrieval finds relevant documents or passages
based on a user's query.

Passage 5:
TF-IDF stands for Term Frequency-Inverse Document Frequency.
It helps identify words that are important in a document
relative to a collection of documents.
